---
title: "Chapter 11: References and BibTeX"
short_title: "11. References"
---

References are handled by Member 3 (`src/reference_parser/`), on the **raw** text of each version, before the
bibliography is removed from the tree.

## 1. Reference sources of a version

| Source | How it is read |
| :--- | :--- |
| `\bibitem` in `thebibliography` | key, authors, title, year, venue, DOI, arXiv ID and URL are extracted from the text of the item |
| amsrefs `\bib{key}{type}{...}` | the named fields are read |
| `.bib` files called by `\bibliography`/`\addbibresource` | BibTeX entries are read; only cited entries are kept (when the paper has citations) |
| `.bbl` file | when a paper ships only a `.bbl` without a `.bib`, the `.bbl` is its only reference source |

Everything is brought to the row format of the `bibtex` table: `citation_key`, `entry_type`, `title`, `authors`,
`year`, `doi`, `journal`, `url`. LaTeX characters are converted to Unicode (`{\"a}` becomes `ä`), and DOIs, arXiv IDs
and URLs are normalized.

In [ ]:
import pandas as pd
import plotly.express as px
from report_utils_m2 import (COLORS, KIND_LABELS, PROBLEM_LABELS, group, human_bytes, human_seconds, load,
                             members, meta, metric, metric_table, num, ordered, pct, percent, ready, secs,
                             show, table)
READY = ready()

In [ ]:
if READY:
    v = load("versions")
    tex = v[v["status"] == "ok"].copy()

    def source(r):
        if r["bbl_input"]:
            return "\\input of a .bbl file"
        if r["has_thebibliography"]:
            return "thebibliography (in the .tex files or read from a .bbl)"
        if r["bib_files"] > 0 and r["bib_fallback"]:
            return ".bib (not declared, .bib files in the folder)"
        if r["bib_files"] > 0:
            return ".bib"
        return "No references"

    tex["Source"] = tex.apply(source, axis=1)
    src = tex.groupby(["member", "Source"]).size().reset_index(name="Versions")
    fig = px.bar(ordered(src), x="member", y="Versions", color="Source", color_discrete_sequence=COLORS,
                 title="Reference source of the versions with TeX", labels={"member": "Range"})
    show(fig)
    t = tex["Source"].value_counts().rename_axis("Source").reset_index(name="Versions")
    t["Share"] = (t["Versions"] / t["Versions"].sum()).map(pct)
    table(t, "Table 11.1. Reference sources, whole group")
    table(metric_table({"Versions reading references from .bbl": "versions_references_from_bbl",
                        "Versions calling a .bib/.bbl missing from the folder": "versions_references_in_missing_bbl"}),
          "Table 11.2. .bbl files")
    if load("bbl").empty:
        print("The exported results contain no fetch_bbl.py log, so no re-downloaded .bbl file is recorded.")

A version may have several sources; Table 11.1 puts it under the first one that applies, in the order `\input` of a
`.bbl`, `thebibliography`, `.bib`. The text of a `.bbl` is merged into the version, so a version reading its `.bbl` is
usually counted under `thebibliography` in Table 11.1; Table 11.2 counts every version that reads one. A version
calling a `.bib`/`.bbl` that is missing from its source has no references; it is counted separately instead of being ignored.

## 2. Deduplicating references across versions (`dedup_bibtex.py`)

The references of every version of a paper are merged. Two entries are **the same reference** when they share a
citation key across versions, a DOI, an arXiv ID or a URL, unless the rest of the entries clearly disagrees, or when
they have the same normalized title, or a near-identical one, with compatible years and first authors. For each group
of duplicates:

- **Field union:** a field present in any entry of the group is kept, instead of choosing one entry and dropping the
  information of the others.
- **One canonical key** is chosen and recorded in `cite_rename_map.json`; every `\cite`-like command of the text is
  renamed to the canonical key (`citation_updater.py`) **before** the tree is built. As a result, two sentences that
  differ only by citation key between two versions become identical and share one `element_id`.

In [ ]:
if READY:
    m, g = members(), group()
    rows = []
    for _, r in pd.concat([m, g.to_frame().T]).iterrows():
        rows.append({"Range": "Group" if r["member"] == "group" else r["member"],
                     "Before deduplication": num(pd.to_numeric(r["bibtex_before"], errors="coerce")),
                     "After deduplication": num(pd.to_numeric(r["bibtex_after"], errors="coerce")),
                     "Reduction": percent(pd.to_numeric(r["bibtex_reduction_percent"], errors="coerce"))})
    table(pd.DataFrame(rows), "Table 11.3. BibTeX entries before and after deduplication")
    papers = load("papers")
    joined = {"Citation key": "joined_by_key", "DOI": "joined_by_doi", "arXiv ID": "joined_by_arxiv", "URL": "joined_by_url",
              "Title": "joined_by_title", "Near-identical title": "joined_by_fuzzy_title"}
    j = pd.DataFrame([{"Matching key": k, "Merges": pd.to_numeric(papers[c], errors="coerce").sum()} for k, c in joined.items()])
    fig = px.bar(j, x="Matching key", y="Merges", text="Merges", color_discrete_sequence=COLORS,
                 title="Which key merged duplicate references")
    show(fig)
    extra = {"Keys renamed to a canonical key": "renamed_keys", "\\cite commands renamed": "cite_commands_renamed",
             "Key conflicts (two different references, same key)": "key_conflicts",
             "Cited keys missing from the bibliography": "cited_not_found",
             "Uncited .bib entries dropped": "bib_uncited_dropped"}
    table(pd.DataFrame([{"Metric": k, "Group": num(pd.to_numeric(papers[c], errors="coerce").sum())} for k, c in extra.items()]),
          "Table 11.4. Citation key renaming")

## 3. Field completeness

In [ ]:
if READY:
    fields = ["title", "authors", "year", "doi", "journal", "url"]
    after = {mm: metric("bibtex_entries_after_dedup", mm) for mm in members()["member"]}
    rows = []
    for f in fields:
        num_, den = 0.0, 0.0
        for mm, n in after.items():
            p = metric(f"bibtex_{f}_completeness_percent", mm)
            if pd.notna(p) and pd.notna(n):
                num_ += p * n
                den += n
        rows.append({"Field": f, "Filled (%)": num_ / den if den else float("nan")})
    fc = pd.DataFrame(rows)
    fig = px.bar(fc, x="Field", y="Filled (%)", text=fc["Filled (%)"].map(lambda x: f"{x:.1f}%"),
                 color_discrete_sequence=COLORS, title="Share of BibTeX entries (after deduplication) with a value in each field")
    fig.update_yaxes(range=[0, 105])
    show(fig)

`title`, `authors` and `year` are the fields Milestone 3 relies on most to match entries with `references.json`;
DOIs and arXiv IDs (in `url`) exist only when the authors wrote them, so lower shares are expected.